# 9.1 订单簿微观结构分析（Order Book）

## 学习目标
- 理解限价委托订单簿（LOB）的结构
- 计算买卖价差（Bid-Ask Spread）和订单流不平衡（OFI）
- 理解 Kyle's Lambda（价格冲击系数）
- 分析逐笔数据的市场微观结构特征


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)
print('Libraries loaded')


## 1. 限价委托订单簿结构

```
卖方（Ask Side）      买方（Bid Side）
价格   数量            价格   数量
100.30  500   ←       100.25  1000
100.28  800   ←       100.22  1500
100.26  1200  ←  ← →  100.20  800
          ↑最优卖价  最优买价↑
          Bid-Ask Spread = 100.26 - 100.25 = 0.01
```

**核心概念**：
- **最优买价（Best Bid）**：市场上最高的买入委托价格
- **最优卖价（Best Ask）**：市场上最低的卖出委托价格
- **中间价（Mid Price）**：(Bid + Ask) / 2
- **价差（Spread）**：Ask - Bid（流动性成本）


In [ ]:
# 模拟一个简单的订单簿
np.random.seed(42)
best_bid = 100.25
best_ask = 100.26
spread = best_ask - best_bid
mid_price = (best_bid + best_ask) / 2

print(f'最优买价（Best Bid）: {best_bid}')
print(f'最优卖价（Best Ask）: {best_ask}')
print(f'中间价（Mid Price）:  {mid_price}')
print(f'价差（Spread）:       {spread} ({spread/mid_price*10000:.2f} bps)')

# 可视化订单簿深度
bid_prices = [best_bid - i*0.01 for i in range(5)]
ask_prices = [best_ask + i*0.01 for i in range(5)]
bid_vol = [1000, 1500, 800, 2000, 1200]
ask_vol = [500, 800, 1200, 600, 900]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(bid_prices, [-v for v in bid_vol], color='green', alpha=0.7, label='买方委托')
ax.barh(ask_prices, ask_vol, color='red', alpha=0.7, label='卖方委托')
ax.axhline(mid_price, color='blue', linestyle='--', lw=2, label=f'中间价={mid_price}')
ax.set_xlabel('委托数量（股）'); ax.set_ylabel('价格')
ax.set_title('限价委托订单簿深度（LOB Depth）')
ax.legend(); plt.tight_layout(); plt.show()


## 2. 订单流不平衡（Order Flow Imbalance, OFI）

$$\text{OFI} = \sum_{t} (\text{买方主动成交量} - \text{卖方主动成交量})$$

OFI > 0 → 买压强 → 短期价格上涨压力
OFI < 0 → 卖压强 → 短期价格下跌压力


In [ ]:
# 模拟 1000 笔逐笔成交数据
np.random.seed(42)
n_trades = 1000

mid_price_start = 100.0
# 随机生成买方/卖方主动成交（1=买方主动, -1=卖方主动）
buy_sell = np.random.choice([1, -1], n_trades, p=[0.52, 0.48])  # 轻微买压
vol = np.random.randint(100, 2000, n_trades)

# 计算价格冲击：每笔成交对中间价的影响
price_impact = 0.0001 * vol * buy_sell
cum_impact = np.cumsum(price_impact)
mid_prices = mid_price_start + cum_impact

# OFI：滚动30笔的净订单流
df = pd.DataFrame({'direction': buy_sell, 'volume': vol, 'mid_price': mid_prices})
df['signed_vol'] = df['direction'] * df['volume']
df['OFI'] = df['signed_vol'].rolling(30).sum()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
df['mid_price'].plot(ax=ax1, color='steelblue', lw=1)
ax1.set_title('模拟中间价走势'); ax1.set_ylabel('价格')
df['OFI'].plot(ax=ax2, color='darkorange', lw=1)
ax2.axhline(0, color='black', lw=0.8); ax2.set_title('订单流不平衡（OFI，30笔滚动）')
plt.tight_layout(); plt.show()


## 🎯 练习

1. 用 OFI 滞后 1 个时间步预测下一步价格变化，计算信号的 IC，OFI 对短期价格有多少预测力？
2. 模拟「大单拆分」场景：一个 50000 股的大单被拆分为 50 个 1000 股的小单，分析其对 OFI 和价格的影响。
3. 研究「Kyle's Lambda」价格冲击系数的估计方法，计算模拟数据中的流动性参数。

---
**下一节** → `02_market_making.ipynb`